# 03 — Fine-tuning do BERTimbau**TCC: Detecção de Smishing em Idosos com Modelos de Linguagem Natural****Modelo:** `neuralmind/bert-base-portuguese-cased` (BERTimbau — NeuralMind)### Decisões de projeto| Parâmetro | Valor | Justificativa ||---|---|---|| Texto de entrada | `texto` original | O BERTimbau é *cased*: maiúsculas carregam informação, e seu tokenizador de subpalavras já lida com a variação que a normalização do notebook 02 remove || `max_length` | 128 tokens | SMS raramente ultrapassa 160 caracteres || Batch size | 16 | Cabe na VRAM da T4 com margem || Épocas | 5 (máx.) | Com early stopping || Learning rate | 2e-5 | Padrão para fine-tuning de BERT || `fp16` | True | Precisão mista: reduz memória e acelera na T4 || Seleção do checkpoint | F2 na validação | Coerente com o custo assimétrico do erro |> **Checkpoints ficam em `/content`, não no Drive.** Cada um tem ~440 MB;> gravá-los via FUSE a cada época é lento e consome a cota de 15 GB. Só o modelo> final é copiado para o Drive.

## 1. Setup

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
# No Colab CADA notebook roda em um runtime próprio: instalar dependências
# em um notebook não vale para os outros. Por isso esta célula se repete em
# todos, e não existe um "notebook de instalação".

REPO = 'https://github.com/SEU-USUARIO/tcc-smishing.git'   # ← ajuste aqui

!git clone -q {REPO} /content/tcc-smishing 2>/dev/null || (cd /content/tcc-smishing && git pull -q)
!pip install -q -r /content/tcc-smishing/requirements.txt

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/tcc-smishing/src')

import config as CFG
CFG.fixar_seeds()
CFG.criar_pastas()
CFG.resumo()

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import evaluation as ev

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "INDISPONÍVEL"}')
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('Troque o runtime para GPU: Ambiente de execução → Alterar tipo.')

treino = pd.read_csv(CFG.SPLIT_FILES['train'], encoding='utf-8')
val    = pd.read_csv(CFG.SPLIT_FILES['val'],   encoding='utf-8')
teste  = pd.read_csv(CFG.SPLIT_FILES['test'],  encoding='utf-8')

# Rótulos para índices: legitima=0, smishing=1
y_treino = treino[CFG.COL_ROTULO].map(CFG.LABEL2ID).values
y_val    = val[CFG.COL_ROTULO].map(CFG.LABEL2ID).values
y_teste  = teste[CFG.COL_ROTULO].map(CFG.LABEL2ID).values

print(f'\nTreino: {len(treino)}  |  Val: {len(val)}  |  Teste: {len(teste)}')
print(f'Positivos no treino: {y_treino.sum()} / {len(y_treino)}')

## 2. Tokenização e dataset

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset

MAX_LENGTH = 128
MODEL_ID = CFG.MODELOS['bertimbau']

tok = AutoTokenizer.from_pretrained(MODEL_ID)


def tokenizar(textos):
    return tok(list(textos), padding=True, truncation=True,
               max_length=MAX_LENGTH, return_tensors='pt')


class DatasetSmishing(Dataset):
    def __init__(self, encodings, rotulos):
        self.encodings = encodings
        self.rotulos = rotulos

    def __len__(self):
        return len(self.rotulos)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.rotulos[i], dtype=torch.long)
        return item


ds_treino = DatasetSmishing(tokenizar(treino[CFG.COL_TEXTO]), y_treino)
ds_val    = DatasetSmishing(tokenizar(val[CFG.COL_TEXTO]),    y_val)
ds_teste  = DatasetSmishing(tokenizar(teste[CFG.COL_TEXTO]),  y_teste)

print(f'Tokens do 1º exemplo: {tok.convert_ids_to_tokens(ds_treino[0]["input_ids"])[:18]} ...')

## 3. Modelo, métricas e perda ponderadaA perda é ponderada pelo inverso da frequência de cada classe. Sem isso o modelotende a favorecer a classe majoritária, o que contraria a priorização do recalldeclarada no projeto.

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import EarlyStoppingCallback
from sklearn.metrics import fbeta_score, recall_score, precision_score
import torch.nn as nn

modelo = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=2, id2label=CFG.ID2LABEL, label2id=CFG.LABEL2ID,
)

# Pesos inversamente proporcionais à frequência
contagem = np.bincount(y_treino, minlength=2)
pesos = torch.tensor(len(y_treino) / (2 * contagem), dtype=torch.float32).to('cuda')
print(f'Contagem por classe: {contagem}  →  pesos: {pesos.tolist()}')

POS_ID = CFG.LABEL2ID[CFG.CLASSE_POSITIVA]


def compute_metrics(eval_pred):
    logits, rotulos = eval_pred
    pred = np.argmax(logits, axis=-1)
    return {
        'f2':        fbeta_score(rotulos, pred, beta=2, pos_label=POS_ID, zero_division=0),
        'f1':        fbeta_score(rotulos, pred, beta=1, pos_label=POS_ID, zero_division=0),
        'recall':    recall_score(rotulos, pred, pos_label=POS_ID, zero_division=0),
        'precisao':  precision_score(rotulos, pred, pos_label=POS_ID, zero_division=0),
    }


class TrainerPonderado(Trainer):
    """Trainer com perda ponderada por classe."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        rotulos = inputs.pop('labels')
        saidas = model(**inputs)
        perda = nn.CrossEntropyLoss(weight=pesos)(saidas.logits, rotulos)
        return (perda, saidas) if return_outputs else perda

## 4. Treinamento

In [ ]:
# Checkpoints no disco LOCAL do Colab, não no Drive
CKPT = '/content/bertimbau_ckpt'

args = TrainingArguments(
    output_dir=CKPT,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',           # em transformers < 4.41 chama-se evaluation_strategy
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f2',      # métrica-título do projeto
    greater_is_better=True,
    fp16=True,
    logging_steps=25,
    save_total_limit=2,
    seed=CFG.SEED,
    report_to='none',
)

trainer = TrainerPonderado(
    model=modelo, args=args,
    train_dataset=ds_treino, eval_dataset=ds_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# resume_from_checkpoint: se a sessão do Colab cair, reexecutar esta célula
# retoma de onde parou em vez de recomeçar do zero
retomar = os.path.isdir(CKPT) and any(d.startswith('checkpoint-') for d in os.listdir(CKPT))
print(f'Retomando de checkpoint: {retomar}')

resultado = trainer.train(resume_from_checkpoint=retomar)
print(f"\nLoss final de treino: {resultado.metrics.get('train_loss', float('nan')):.4f}")

In [ ]:
# Curvas de aprendizado
historico = trainer.state.log_history
logs_treino = [e for e in historico if 'loss' in e and 'eval_loss' not in e]
logs_val    = [e for e in historico if 'eval_loss' in e]

if logs_treino and logs_val:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot([e['step'] for e in logs_treino], [e['loss'] for e in logs_treino],
                 marker='.', label='Treino')
    axes[0].plot([e['step'] for e in logs_val], [e['eval_loss'] for e in logs_val],
                 marker='o', label='Validação')
    axes[0].set_title('Loss')
    axes[0].set_xlabel('Passo')
    axes[0].legend()

    epocas = [e['epoch'] for e in logs_val]
    for chave, estilo in [('eval_f2', '-o'), ('eval_recall', '--s'), ('eval_precisao', ':^')]:
        if chave in logs_val[0]:
            axes[1].plot(epocas, [e[chave] for e in logs_val], estilo, label=chave.replace('eval_', ''))
    axes[1].set_title('Métricas na validação')
    axes[1].set_xlabel('Época')
    axes[1].set_ylim(0, 1)
    axes[1].legend()

    plt.suptitle('BERTimbau — curvas de aprendizado')
    plt.tight_layout()
    plt.savefig(f"{CFG.PATHS['figures']}/03_bert_curvas.png", dpi=150, bbox_inches='tight')
    plt.show()

## 5. Calibração do limiar na validação

In [ ]:
def pontuacoes(dataset):
    """Probabilidade da classe positiva para cada exemplo."""
    logits = trainer.predict(dataset).predictions
    return torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, POS_ID]


score_val = pontuacoes(ds_val)
y_val_str = val[CFG.COL_ROTULO].values

limiar, f2_val = ev.calibrar_limiar(y_val_str, score_val, beta=2)

m05 = ev.calcular_metricas(y_val_str, ev.aplicar_limiar(score_val, 0.5), score_val)
mca = ev.calcular_metricas(y_val_str, ev.aplicar_limiar(score_val, limiar), score_val)

print(f'Limiar escolhido: {limiar:.4f}')
print(f'  val @0.5       F2={m05["f2"]:.4f}  recall={m05["recall"]:.4f}  FN={m05["FN"]}')
print(f'  val @calibrado F2={mca["f2"]:.4f}  recall={mca["recall"]:.4f}  FN={mca["FN"]}')

## 6. Avaliação no teste e salvamento

In [ ]:
score_teste = pontuacoes(ds_teste)
y_teste_str = teste[CFG.COL_ROTULO].values
pred = ev.aplicar_limiar(score_teste, limiar)

metricas = ev.calcular_metricas(y_teste_str, pred, score_teste)
print('=== BERTimbau — teste ===')
for chave in ev.ORDEM_METRICAS:
    print(f'  {ev.NOMES_METRICAS[chave]:16}: {metricas[chave]:.4f}')
print(f"  VP={metricas['VP']}  FN={metricas['FN']}  FP={metricas['FP']}  VN={metricas['VN']}")

ev.salvar_predicoes(teste['id'], y_teste_str, pred, score_teste, 'bertimbau')

ev.plot_confusao(y_teste_str, pred,
                 f'BERTimbau\nF2={metricas["f2"]:.3f}  Recall={metricas["recall"]:.3f}',
                 salvar_como='03_confusao_bertimbau.png')
plt.show()

In [ ]:
# Só o modelo final vai para o Drive — os checkpoints ficam no disco local
DESTINO = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "notebooks")
trainer.save_model(DESTINO)
tok.save_pretrained(DESTINO)

import json
with open(f'{DESTINO}/limiar.json', 'w') as f:
    json.dump({'limiar': float(limiar), 'criterio': 'F2 na validação'}, f, indent=2)

print(f'Modelo salvo em: {DESTINO}')
print(os.listdir(DESTINO))
print('\nProssiga para o notebook 04_llm_llama.ipynb.')